# 🛰️ 衛星データ解析（Python実装）練習ノートブック
スマートパン屋農家「探究コーナー：衛星データ解析」と同じ計算を、自分の手で動かします。

**3段構え**：① 現象を計算する → ② 数値を可視化する → ③ 自分の条件で判断する

- 1〜5 節は **架空の地図**（三原の小麦畑と川沿いを想定。実測ではありません）で、ネットがなくても最後まで動きます。
- 6 節は任意：ネットにつながる環境で、Sentinel-2 の無料公開データを読み込む例です。
- 下敷き：『SAR衛星データ解析入門』（講談社）・『Pythonで学ぶ衛星データ解析基礎』。コードは独自のもので、書籍の転載ではありません。

In [ ]:
# 0. 準備：リポジトリの計算部品（satellite_lab.py）を使えるようにする
import os, sys
if not os.path.exists("satellite_lab.py"):
    if not os.path.exists("Smart-Bakery-Farmer_20260923"):
        os.system("git clone --depth 1 https://github.com/nayukaya3-cn/Smart-Bakery-Farmer_20260923.git")
    sys.path.insert(0, "Smart-Bakery-Farmer_20260923")
import numpy as np
import matplotlib.pyplot as plt
import satellite_lab as sl
plt.rcParams["figure.dpi"] = 110
sc = sl.make_scene(seed=0, cloud_cover=0.25, flood_m=1.5)
print("地図の大きさ:", sc.cls.shape, "／ 1画素 =", sl.PIXEL_HA, "ha")

## 1. 光学：バンド演算で NDVI を計算する
植物の葉は赤い光を吸い、近赤外をよく反射します。NDVI = (近赤外 − 赤) / (近赤外 + 赤)。

In [ ]:
red, nir = sc.optical[2], sc.optical[3]
nd = sl.ndvi(red, nir)
true_color = np.clip(np.stack([sc.optical[2], sc.optical[1], sc.optical[0]], -1) * 3.5, 0, 1)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(true_color); ax[0].set_title("true color")
im = ax[1].imshow(nd, cmap="RdYlGn", vmin=-0.5, vmax=0.9); ax[1].set_title("NDVI")
plt.colorbar(im, ax=ax[1]); [a.axis("off") for a in ax]; plt.show()
for c, name in enumerate(sl.CLASSES):
    m = (sc.cls == c) & ~sc.cloud
    print(f"{name:6s} NDVI平均 {nd[m].mean():+.2f}")
print("雲の画素の NDVI 平均:", round(float(nd[sc.cloud].mean()), 2), "← 雲を『植物がない』と読まない")

## 2. SAR：dB・スペックル・Lee フィルタ・大津の方法で水域

In [ ]:
raw_db = sl.to_db(sc.sar_before)
filt_db = sl.to_db(sl.lee_filter(sc.sar_before, r=2, looks=sc.looks))
thr = sl.water_threshold(filt_db)   # 大津の方法を2段階で
print(f"水の閾値（自動）: {thr:.1f} dB")
truth = sc.cls == sl.WATER
for name, img in [("生のまま", raw_db), ("ならした後", filt_db)]:
    s = sl.scores(truth, img < thr)
    print(f"{name}: F1={s['F1']:.2f} 誤報={s['誤報']} 見逃し={s['見逃し']}")
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(raw_db, cmap="gray", vmin=-28, vmax=2); ax[0].set_title("raw (dB)")
ax[1].imshow(filt_db, cmap="gray", vmin=-28, vmax=2); ax[1].set_title("Lee filter")
ax[2].hist(filt_db.ravel(), bins=120); ax[2].axvline(thr, color="r"); ax[2].set_title("histogram")
[a.axis("off") for a in ax[:2]]; plt.show()

## 3. 変化を見つける：大雨の前後の差で「新しく浸水した所」
③ 判断：自分の圃場（オレンジの枠）が何％浸かったら行動するかを、自分で決めます。

In [ ]:
diff, flooded = sl.change_map(sc.sar_before, sc.sar_after, thr_drop_db=3, water_thr_db=thr, looks=sc.looks)
print("前後の差:", {k: round(v, 2) for k, v in sl.scores(sc.flood, flooded).items() if k in ("再現率", "適合率", "F1")})
print("後の1枚だけ:", {k: round(v, 2) for k, v in sl.scores(sc.flood, sl.water_map(sc.sar_after, thr)).items() if k in ("再現率", "適合率", "F1")})
mine = sl.crop(flooded, sc.my_field)
ACTION_PERCENT = 10   # ← 自分の基準に変える
print(f"わたしの圃場：{mine.size * sl.PIXEL_HA:.2f} ha のうち {mine.mean():.0%} が浸水と判定")
print("判定:", "行動する" if mine.mean() * 100 >= ACTION_PERCENT else "見守る")
r0, r1, c0, c1 = sc.my_field
plt.imshow(diff, cmap="RdBu", vmin=-12, vmax=12)
plt.gca().add_patch(plt.Rectangle((c0 - .5, r0 - .5), c1 - c0, r1 - r0, fill=False, color="orange", lw=2))
plt.title("after - before (dB)"); plt.colorbar(); plt.axis("off"); plt.show()
# 水位を変えて、自分の圃場の「デッドライン」を探す
for m in [1.0, 1.5, 2.0, 2.5, 3.0]:
    s2 = sl.make_scene(0, flood_m=m)
    print(f"水位 +{m} m → 圃場の浸水 {sl.crop(s2.flood, s2.my_field).mean():.0%}")

## 4. 分類：教師あり（ナイーブベイズ）と教師なし（k-means）
正解率は **テスト側の画素だけ** で数えます。分け方（ランダム／場所で分ける）も変えてみましょう。

In [ ]:
cols = ["赤", "近赤外", "短波長赤外", "NDVI", "NDWI", "VV dB（ならし後）"]
for how in ["場所で分ける", "ランダム"]:
    R = sl.classify(sc, cols, how=how, n_train=400)
    print(how, {k: f"{v:.1%}" for k, v in R["正解率"].items()})
R = sl.classify(sc, cols)
print(R["混同行列"])

## 5. 言説と観測：船を CFAR で数え、乖離指標 D を出す
D が大きいことは **確かめる合図** であって、誰かの意図の証明ではありません。

In [ ]:
wk = sl.weekly_port(weeks=12, base=20, real_drop=0.2, claim_drop=0.8, change_week=7, seed=0)
print(wk)
g = sl.gap_index(wk, change_week=7)
print({k: round(v, 2) for k, v in g.items()})
plt.plot(wk["週"], wk["言説が示す隻数"], "r-", label="claim")
plt.plot(wk["週"], wk["SARで数えた隻数"], "b-o", label="SAR count")
plt.plot(wk["週"], wk["本物の隻数（ふつうは分からない）"], "k:", label="truth")
plt.legend(); plt.xlabel("week"); plt.show()

## 5.5 3層の観測をつなぐ：衛星（広域）→ ドローン・地上ロボット（圃場）→ 目視・スマホ（地点）
見る範囲が広いほど更新はゆっくり、いまに近いほど見える範囲は狭い。3つを **ベイズの定理（対数オッズの足し算）** で1枚の浸水確率の地図に重ねます。
- 衛星が水位の上がりきる前に来ると、古い「乾いている」で判断を誤る → 割り引く／ドローンで確かめる
- ドローンは「迷っている所」へ向けると効く
- 衛星があると、古い写真・別の場所の写真（うわさ）に強くなる

In [ ]:
now = sl.make_scene(0, flood_m=2.0)        # いまの本当の様子
early = sl.make_scene(0, flood_m=1.5)      # 衛星が来たのは、水位があと 0.5 m 低いとき
for title, kw in [("その日に衛星が来た", {}),
                  ("衛星が早く来た", {"sat_scene": early}),
                  ("衛星が早く来た＋割り引く＋圃場へドローン", {"sat_scene": early, "sat_weight": 0.3, "drone": "わたしの圃場"}),
                  ("スマホだけ・うわさが多い", {"sat_weight": 0, "drone": "飛ばさない", "n_phone": 60, "trust": 0.6, "stale": 0.5, "assumed_trust": 0.97}),
                  ("衛星あり・うわさが多い", {"drone": "飛ばさない", "n_phone": 60, "trust": 0.6, "stale": 0.5, "assumed_trust": 0.97})]:
    L = sl.layered(now, level_m=1.5, **kw)
    print(title, "ブライア:", L["scores"]["ブライアスコア"].round(3).tolist(),
          "／ わたしの圃場の見積もり(ha):", L["scores"]["わたしの圃場：浸水面積の見積もり(ha)"].round(2).tolist(),
          "／ 本当:", round(L["わたしの圃場：本当の浸水(ha)"], 2))
L = sl.layered(now, level_m=1.5)
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for a, (name, pm) in zip(ax, L["maps"].items()):
    a.imshow(np.where(now.cls == sl.FARM, pm, np.nan), cmap="Blues", vmin=0, vmax=1); a.set_title(name[:2]); a.axis("off")
plt.show()

## 6.（任意）実データ：Sentinel-2 で自分の地域の NDVI
- ネットにつながる環境（Google Colab など）で動かします。オフラインや学校のネットワークでは失敗することがあります。
- 無料の公開カタログ（Element 84 Earth Search）から雲の少ない日を1つ選び、赤と近赤外で NDVI を計算します。
- `BBOX` は **大まかな例** です。地図で自分の地域の経度・緯度を確かめて書き換えてください（番地レベルの位置を人に見せる資料には載せない）。
- 日本の衛星データは **Tellus** でも探せます（アカウント登録が必要）。

In [ ]:
RUN_REAL_DATA = False   # ← True にすると実行
BBOX = (132.95, 34.48, 133.10, 34.58)   # (西経度, 南緯度, 東経度, 北緯度) の例：三原市の北部あたり。要確認
DATES = "2026-04-01/2026-05-31"         # 小麦の茎立ち〜出穂のころ
if RUN_REAL_DATA:
    os.system("pip -q install pystac-client rioxarray")
    from pystac_client import Client
    import rioxarray
    cat = Client.open("https://earth-search.aws.element84.com/v1")
    items = list(cat.search(collections=["sentinel-2-l2a"], bbox=BBOX, datetime=DATES,
                            query={"eo:cloud_cover": {"lt": 20}}, max_items=10).items())
    if not items:
        print("雲の少ない日が見つかりませんでした。DATES を広げてください。")
    else:
        item = min(items, key=lambda i: i.properties["eo:cloud_cover"])
        print("使う画像:", item.id, item.datetime, "雲", item.properties["eo:cloud_cover"], "%")
        band = lambda k: rioxarray.open_rasterio(item.assets[k].href).rio.clip_box(*BBOX, crs="EPSG:4326").squeeze().astype(float)
        red_r, nir_r = band("red"), band("nir")
        nd_r = sl.ndvi(red_r.values, nir_r.values)
        plt.imshow(nd_r, cmap="RdYlGn", vmin=-0.2, vmax=0.9); plt.colorbar(); plt.title(f"NDVI {item.datetime:%Y-%m-%d}")
        plt.axis("off"); plt.show()

## 振り返り（3段構え）
1. **計算**：どの式・どの閾値で判定したか
2. **可視化**：どのグラフ・地図で確かめたか
3. **判断**：自分の圃場・自分の生活で、何を・いつ・どうするか（数字を1つ以上入れる）

⚠️ 計算は教育用の単純化モデルです。実際の避難判断は、自治体のハザードマップ・気象庁の情報・避難指示に従ってください。